# BusinessGPT v16 Fine-Tuning: Qwen3.5-**9B** (abliterated) — LoRA

Self-contained Kaggle notebook: preprocessing + fine-tuning with LoRA + completion-only loss.

**v16 changes vs v15** (data-only — same 9B base, same hyperparams):
- New: Qwen-72B distilled SFT augment (`eval/distilled_qwen72b_v16.jsonl`, ~700-1000 examples after style filter), loaded alongside `sft_augment.jsonl` with `DISTILL_REPEAT=1`. Goal: inject frontier-model reasoning quality into in-distribution chat contexts.
- ORPO and reward-model best-of-N are downstream of this notebook; see `orpo.ipynb` and `reward_model.ipynb`.

**v15 changes vs v14**: model size jump 2B → **9B** (huihui-ai/Huihui-Qwen3.5-9B-abliterated).
- Diagnosis from v14 pairwise: style memorization is saturated, but reasoning (connecting style to meaningful replies) is the bottleneck. That's compute-bound — solved by larger params, not more data.
- Same Qwen3.5 family → drop-in pipeline, same chat template, same hybrid linear_attn DPO concerns (already characterized in v13-dpo runs).
- 9B Q5_K_M GGUF ≈ 6.4 GB, fits the new 12 GB CPU RAM constraint with comfortable KV-cache headroom.
- Memory tuning for T4×2 training: MAX_SEQ_LENGTH 4096 → 2048, GRAD_ACCUM 8 → 16. 9B fp16 = 18 GB across 2 GPUs is tight but works with gradient checkpointing.
- **Inline eval disabled** — won't fit in 12 h Kaggle budget on 9B. Run `eval_only.ipynb` separately after this notebook completes.

**v14 changes vs v13** (data-only, hyperparams unchanged):
- New: SFT augmentation from labeled preference pairs. `eval/sft_augment.jsonl` (1144 unique chosen-only examples, super tier 2x replicated) is loaded as additional training data. AUGMENT_REPEAT=2 — each example appears 2x, super 4x.
- Sidesteps DPO instability discovered on v13→v13-dpo (NaN collapse + soft entropy collapse). Per-token CE loss is length-bias-immune and has no off-policy log-ratio.

**v13 changes vs v12** (data-only, hyperparams unchanged):
- `TARGET_RATIO` 0.15 → 0.10
- Format C (pure trigger → full window) weight 33% → 10%; weights now A=40% / B=50% / C=10%
- 13 → 8 trigger templates: dropped generic ones; kept only verb-anchored (`зачитай`, `кинь трек`, `спой`, `читни` etc.)
- `window` 8 → 5: max assistant output capped at 5 lines.
- Newline normalization: 50% chance to join lyric lines with `". "` instead of `"\n"`.

**v12 changes vs v11**:
- Drop Telegram "gay test" bot spam (was poisoning ~3% of targets).
- Rap ratio 5% → 15%, plus chat-style triggers.

**Requirements**: Kaggle GPU T4×2 (≥ 24 GB VRAM total)

## 0. Install Dependencies

In [ ]:
%pip install vllm --torch-backend=auto --extra-index-utorl https://wheels.vllm.ai/nightly
%pip install git+https://github.com/huggingface/transformers.git

In [ ]:
%%capture
%pip install -U huggingface_hub trl kagglehub datasets accelerate bitsandbytes torchao
%pip install git+https://github.com/huggingface/peft.git

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
!hf auth list

## 1. Load & Preprocess Data

In [ ]:
import json
import glob
import os
from collections import Counter
from datetime import datetime

# --- Load raw data ---
# On Kaggle: attach dataset "alextech123/businessraw" to notebook,
# it will be at /kaggle/input/businessraw/result.json
KAGGLE_INPUT = "/kaggle/input/businessraw/result.json"

if os.path.exists(KAGGLE_INPUT):
    json_path = KAGGLE_INPUT
    print(f"Using Kaggle input: {json_path}")
else:
    import kagglehub
    dataset_path = kagglehub.dataset_download("alextech123/businessraw")
    json_files = glob.glob(f"{dataset_path}/**/*.json", recursive=True)
    json_path = json_files[0]
    print(f"Downloaded via kagglehub: {json_path}")

with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

raw_messages = data["messages"]
print(f"Total raw messages: {len(raw_messages)}")

In [ ]:
# --- Flatten text field ---
import re as _re

def flatten_text(text_field):
    if isinstance(text_field, str):
        return text_field
    if isinstance(text_field, list):
        parts = []
        for part in text_field:
            if isinstance(part, str):
                parts.append(part)
            elif isinstance(part, dict) and "text" in part:
                parts.append(part["text"])
        return "".join(parts)
    return ""

# --- Extract records ---
records = []
for msg in raw_messages:
    if msg.get("type") != "message":
        continue
    records.append({
        "id": msg["id"],
        "from": msg.get("from", "Unknown"),
        "text": flatten_text(msg.get("text", "")),
        "timestamp": int(msg.get("date_unixtime", 0)),
        "reply_to_message_id": msg.get("reply_to_message_id"),
    })

# --- Find BusinessGPT introduction date ---
BOT_NAMES = {"BusinessGPT"}
bot_timestamps = [r["timestamp"] for r in records if r["from"] in BOT_NAMES]
bot_introduction_ts = min(bot_timestamps) if bot_timestamps else float("inf")

bot_intro_date = datetime.fromtimestamp(bot_introduction_ts)
print(f"BusinessGPT first appeared: {bot_intro_date.strftime('%Y-%m-%d %H:%M')}")
print(f"Total records: {len(records)}")

# --- Split by bot introduction date ---
records_pre = [r for r in records if r["timestamp"] < bot_introduction_ts]
records_post = [r for r in records if r["timestamp"] >= bot_introduction_ts]
print(f"Pre-bot records: {len(records_pre)}, Post-bot records: {len(records_post)}")

# --- Standard filtering (both halves) ---
bot_msg_ids = {r["id"] for r in records if r["from"] in BOT_NAMES}

# Bot output that leaked under user names (Telegram /slash commands return as
# the invoker — same pattern across many senders is the giveaway). Discovered
# via eval/scan_bot_patterns.py — re-run after each data refresh to surface
# new bots that joined the chat.
_BOT_LEAK_PATTERNS = [
    _re.compile(r"I am \d+% gay", _re.IGNORECASE),       # 'gay test' bot, 461 msgs / 11 senders
    _re.compile(r"^/degenerate\b"),                      # /degenerate bot, 44 msgs
    _re.compile(r"^/threshold\s+\d"),                    # /threshold N bot, 13 msgs
    _re.compile(r"[ㅤᅟᅠ⠀]{3,}"),      # invisible whitespace spam (Hangul filler), 18 msgs
    _re.compile(r"^(\d)\1{6,}$"),                         # digit spam '2222222222', 11 msgs
]

def standard_filter(recs):
    """Drop empty text, named bot messages, @mentions, /generate, replies to bot, known bot-leak patterns."""
    out = []
    for r in recs:
        if not r["text"].strip() or r["from"] in BOT_NAMES:
            continue
        text_lower = r["text"].lower()
        if "@businessgpt_text_bot" in text_lower or "/generate" in text_lower:
            continue
        if any(p.search(r["text"]) for p in _BOT_LEAK_PATTERNS):
            continue
        if r.get("reply_to_message_id") in bot_msg_ids:
            continue
        out.append(r)
    return out

filtered_pre = standard_filter(records_pre)

# --- Enhanced filtering for post-bot data: also scrub bot name references ---
BOT_MENTION_PATTERNS = [
    r"businessgpt", r"бизнесгпт", r"бизнес\s*gpt", r"business\s*gpt",
    r"бизнес\s*бот", r"гпт\s*бот",
]
_bot_mention_re = _re.compile("|".join(BOT_MENTION_PATTERNS), _re.IGNORECASE)

def enhanced_filter(recs):
    """Standard filter + drop messages that mention the bot by name."""
    base = standard_filter(recs)
    return [r for r in base if not _bot_mention_re.search(r["text"])]

filtered_post = enhanced_filter(records_post)

# --- Stats: bot-leak dropped breakdown ---
bot_leak_dropped_pre = sum(1 for r in records_pre if any(p.search(r["text"]) for p in _BOT_LEAK_PATTERNS))
bot_leak_dropped_post = sum(1 for r in records_post if any(p.search(r["text"]) for p in _BOT_LEAK_PATTERNS))
print(f"Bot-leak dropped: {bot_leak_dropped_pre} pre-bot, {bot_leak_dropped_post} post-bot")
print(f"Filtered pre-bot: {len(filtered_pre)}, Filtered post-bot: {len(filtered_post)}")

In [ ]:
# --- Merge consecutive messages from same sender ---
def merge_consecutive(messages, max_gap_sec=60):
    merged = []
    for msg in messages:
        if (
            merged
            and merged[-1]["from"] == msg["from"]
            and msg["timestamp"] - merged[-1]["timestamp"] < max_gap_sec
        ):
            merged[-1]["text"] += "\n" + msg["text"]
            merged[-1]["timestamp"] = msg["timestamp"]
        else:
            merged.append(dict(msg))
    return merged

merged_pre = merge_consecutive(filtered_pre, max_gap_sec=60)
unique_pre = {m["from"] for m in merged_pre}
print(f"Pre-bot:  {len(filtered_pre)} -> {len(merged_pre)} turns ({len(unique_pre)} users)")

merged_post = merge_consecutive(filtered_post, max_gap_sec=60)
unique_post = {m["from"] for m in merged_post}
print(f"Post-bot: {len(filtered_post)} -> {len(merged_post)} turns ({len(unique_post)} users)")

all_names = unique_pre | unique_post
print(f"All senders: {sorted(all_names)}")

In [ ]:
# --- Split into sessions (pre-bot only; post-bot data is spam/slop) ---
MIN_SESSION_LEN = 5

def split_sessions(messages, gap_threshold_sec=3600):
    sessions, current = [], []
    for msg in messages:
        if current and msg["timestamp"] - current[-1]["timestamp"] > gap_threshold_sec:
            if len(current) >= MIN_SESSION_LEN:
                sessions.append(current)
            current = []
        current.append(msg)
    if len(current) >= MIN_SESSION_LEN:
        sessions.append(current)
    return sessions

sessions = split_sessions(merged_pre, gap_threshold_sec=3600)
msg_counts = [len(s) for s in sessions]
print(f"Sessions: {len(sessions)} ({sum(msg_counts)} msgs)")
print(f"  Session sizes: min={min(msg_counts)}, median={sorted(msg_counts)[len(msg_counts)//2]}, "
      f"max={max(msg_counts)}, mean={sum(msg_counts)/len(msg_counts):.1f}")

In [ ]:
# --- Create multi-turn training examples with data augmentation ---
SYSTEM_PROMPT = (
    "Ты BusinessGPT. Пиши как студент в мессенджере: коротко, дерзко, ахуевше, по-пидорски. "
    "Часто вставляй слова-паразиты: бля, нах, блять, ёпт, пиздец."
)

MIN_RESPONSE_LEN = 3
MAX_RESPONSE_LEN = 300

import re as _quality_re

_EMOJI_ONLY = _quality_re.compile(
    r"^[\s\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF"
    r"\U0001F900-\U0001F9FF\U00002702-\U000027B0\U0000FE00-\U0000FE0F"
    r"\U0000200D\U00002640\U00002642\U0000231A-\U0000231B]+$"
)

_CMD_RE = _quality_re.compile(r"^/\w")

def _is_quality_response(text):
    if len(text) < MIN_RESPONSE_LEN or len(text) > MAX_RESPONSE_LEN:
        return False
    if _EMOJI_ONLY.match(text):
        return False
    if _CMD_RE.match(text):
        return False
    return True


def create_multiturn_examples(sessions, window_sizes=(5, 10, 15), min_context=2):
    """Multi-turn examples with proper user/assistant structure.

    Target user's prior messages become assistant turns (teaches style),
    everyone else becomes user turns with name prefixes.
    Multiple window sizes per target provide data augmentation.
    """
    examples = []
    for session in sessions:
        for i in range(min_context, len(session)):
            target = session[i]
            response = target["text"].strip()
            if not response or not _is_quality_response(response):
                continue
            target_user = target["from"]

            prev_start = None
            for ws in sorted(window_sizes):
                start = max(0, i - ws)
                if start == prev_start:
                    continue
                prev_start = start
                context = session[start:i]
                if len(context) < min_context:
                    continue

                messages = [{"role": "system", "content": SYSTEM_PROMPT}]

                for j, msg in enumerate(context):
                    is_last_ctx = (j == len(context) - 1)
                    if msg["from"] == target_user and not is_last_ctx:
                        role = "assistant"
                        content = msg["text"]
                    else:
                        role = "user"
                        content = f"{msg['from']}: {msg['text']}"

                    if len(messages) == 1 and role == "assistant":
                        role = "user"
                        content = f"{msg['from']}: {msg['text']}"

                    if messages[-1]["role"] == role:
                        messages[-1]["content"] += "\n" + content
                    else:
                        messages.append({"role": role, "content": content})

                messages.append({"role": "assistant", "content": response})
                examples.append({"messages": messages})

    return examples


import random
random.seed(42)

all_examples = create_multiturn_examples(sessions, window_sizes=(10, 15), min_context=2)
random.shuffle(all_examples)

n_turns = [len(ex["messages"]) for ex in all_examples]
n_multiturn = sum(1 for n in n_turns if n > 3)
print(f"Total chat examples: {len(all_examples)}")
print(f"  Multi-turn (>3 msgs): {n_multiturn} ({n_multiturn/len(all_examples)*100:.1f}%)")
print(f"  Turn counts: min={min(n_turns)}, median={sorted(n_turns)[len(n_turns)//2]}, max={max(n_turns)}")

samples = random.sample(all_examples, min(5, len(all_examples)))
for idx, ex in enumerate(samples):
    print(f"\n{'='*60}")
    print(f"Example {idx+1} ({len(ex['messages'])} turns)")
    for m in ex["messages"]:
        tag = m["role"][0].upper()
        preview = m["content"].replace('\n', ' | ')[:100]
        print(f"  [{tag}] {preview}")
print(f"\n{'='*60}")

In [ ]:
# --- Load and preprocess Russian rap lyrics ---
import pandas as pd

RAP_DATA_DIR = "/kaggle/input/datasets/wadzim/modern-russian-rap"
rap_csvs = glob.glob(os.path.join(RAP_DATA_DIR, "*.csv"))
assert rap_csvs, f"No CSV files found in {RAP_DATA_DIR}"
rap_df = pd.concat([pd.read_csv(f) for f in rap_csvs], ignore_index=True)
print(f"Loaded rap dataset: {len(rap_df)} songs, columns: {list(rap_df.columns)}")

SELECTED_ARTISTS = {
    "Pharaoh", "Элджей", "Скриптонит", "Пошлая Молли",
    "ЛСП", "Гуф", "АК-47", "Yanix", "Oxxxymiron", 
    "MORGENSHTERN", "Lizer", "Kizaru",
    "GONE.Fludd", "Дора", "INSTASAMKA", "Платина",
    "FRIENDLY THUG 52 NGG", "Boulevard Depo", "SALUKI",
}
selected_lower = {a.lower().strip() for a in SELECTED_ARTISTS}

def artist_match(row):
    for col in ["Artist", "Genius Artist Name"]:
        val = str(row.get(col, "")).strip().lower()
        if val in selected_lower:
            return True
    return False

rap_df = rap_df[rap_df.apply(artist_match, axis=1)].copy()
rap_df = rap_df.dropna(subset=["Lyric"])
rap_df = rap_df[rap_df["Lyric"].str.strip().astype(bool)]

found_artists = set()
for _, row in rap_df.iterrows():
    for col in ["Artist", "Genius Artist Name"]:
        val = str(row.get(col, "")).strip().lower()
        if val in selected_lower:
            found_artists.add(val)
missing = selected_lower - found_artists
print(f"Filtered to {len(rap_df)} songs from {len(found_artists)} artists")
if missing:
    print(f"  WARNING: Not found in dataset: {missing}")

import re as _re

def split_lines(text):
    """Split lyrics into lines. Handles newlines, or period/sentence-based formats."""
    lines = _re.split(r"\r?\n|\r", text)
    lines = [l.strip() for l in lines if l.strip()]
    if len(lines) > 1:
        return lines
    parts = _re.split(r"(?<=[.!?])\s*(?=[А-ЯA-Z«\"])", text)
    lines = [p.strip().rstrip(".") for p in parts if p.strip()]
    return [l for l in lines if len(l) > 1]

sample_lyric = rap_df.iloc[0]["Lyric"]
sample_lines = split_lines(sample_lyric)
print(f"Sample lyric: {len(sample_lyric)} chars -> {len(sample_lines)} lines")
for i, l in enumerate(sample_lines[:8]):
    print(f"  {i+1}. {l[:80]}")
if len(sample_lines) > 8:
    print(f"  ... ({len(sample_lines)} total)")

line_counts = rap_df["Lyric"].apply(lambda x: len(split_lines(x)))
print(f"\nLines per song: min={line_counts.min()}, median={line_counts.median():.0f}, "
      f"max={line_counts.max()}, mean={line_counts.mean():.1f}")
print(f"Songs with >= 5 lines: {(line_counts >= 5).sum()} / {len(rap_df)}")

# --- Chat-style rap triggers (v13) ---
# v12 generalised too aggressively: short prompts like "столица австралии?"
# triggered 8-line lyric responses because format C (trigger only -> full window)
# was 33% and trigger templates included generic ones without an explicit
# read/sing verb.
# v13 fixes (data-only, all in this cell):
#   1) Drop generic trigger templates ({artist}?, {artist} бля, давай {artist}, ...)
#      keeping only verb-anchored ones (8 instead of 13).
#   2) Format C weight 33% -> 10%; A=40% / B=50% / C=10%.
#   3) TARGET_RATIO 0.15 -> 0.10.
#   4) window 8 -> 5: max assistant output is 5 lines (was 8). Labelers flagged
#      8-line responses as "перебор". A/B split 2/3, C=full 5 lines.
#   5) Newline normalization in assistant content: 50% chance to join lines with
#      ". " instead of "\n". v12 model leaked rap-style newlines into chat
#      replies because every rap example used \n as line separator.
RAP_CHAT_NAMES = [
    "Некит Русанов", "Егориус", "A. H.", "Александр Блок", "Саня Блок",
    "Булгак", "Влад Блок", "Старый Мельник", "Вован Крюк",
]
RAP_TRIGGER_TEMPLATES = [
    "зачитай {artist}",
    "кинь трек {artist}",
    "спой {artist}",
    "трек от {artist} пж",
    "зачитать {artist}?",
    "{artist} читни",
    "а ну {artist} ебани",
    "мне тут {artist} напомнили",
]


def _make_rap_example(user_content, assistant_content):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ],
    }


def _chat_trigger(artist):
    """Synthetic chat message asking for a track from `artist`."""
    name = random.choice(RAP_CHAT_NAMES)
    trigger = random.choice(RAP_TRIGGER_TEMPLATES).format(artist=artist)
    return f"{name}: {trigger}"


def _join_lyric(lines):
    """Join lyric lines, randomly choosing newlines or sentence-style punctuation.

    50/50 split: half of rap examples preserve the song's line-break structure,
    half are joined with ". " as if spoken inline. This stops the model from
    locking into "rap == newline-heavy" output that bleeds into chat replies.
    """
    if len(lines) <= 1:
        return lines[0] if lines else ""
    if random.random() < 0.5:
        return ". ".join(l.rstrip(".,!? ") for l in lines)
    return "\n".join(lines)


def chunk_lyrics(lyric_text, artist, window=5, stride=3, min_window=4):
    """Produce rap SFT examples in three formats:

      A (40%) `[Artist]\\n<2 lines>`            -> 3-line continuation
      B (50%) `{name}: {trigger} {artist}\\n<2 lines>` -> 3-line continuation (bridge)
      C (10%) `{name}: {trigger} {artist}`     -> 5-line response (pure request)

    Assistant content is joined via `_join_lyric()` (50/50 ". " vs "\\n").
    """
    lines = split_lines(lyric_text)
    examples = []

    def _one_chunk(chunk):
        r = random.random()
        if r < 0.40:
            user_content = f"[{artist}]\n" + "\n".join(chunk[:2])
            assistant_content = _join_lyric(chunk[2:])
        elif r < 0.90:
            user_content = _chat_trigger(artist) + "\n" + "\n".join(chunk[:2])
            assistant_content = _join_lyric(chunk[2:])
        else:
            user_content = _chat_trigger(artist)
            assistant_content = _join_lyric(chunk)
        return _make_rap_example(user_content, assistant_content)

    if len(lines) >= window:
        for start in range(0, len(lines) - window + 1, stride):
            examples.append(_one_chunk(lines[start:start + window]))
    elif len(lines) >= min_window:
        examples.append(_one_chunk(lines))
    return examples


rap_examples = []
for _, row in rap_df.iterrows():
    artist = str(row.get("Artist", "")).strip()
    rap_examples.extend(chunk_lyrics(row["Lyric"], artist))

TARGET_RATIO = 0.10
max_rap = int(len(all_examples) * TARGET_RATIO / (1 - TARGET_RATIO))
if len(rap_examples) > max_rap:
    random.seed(42)
    rap_examples = random.sample(rap_examples, max_rap)
    print(f"Subsampled rap examples to {max_rap} (target ratio {TARGET_RATIO*100:.0f}%)")

total = len(all_examples) + len(rap_examples)
ratio = len(rap_examples) / total * 100 if total > 0 else 0
print(f"\nChat examples: {len(all_examples)}")
print(f"Rap examples:  {len(rap_examples)}")
print(f"Total:         {total} (rap: {ratio:.1f}%)")

if rap_examples:
    print(f"\nSample rap examples (format & join variety):")
    for idx in range(min(4, len(rap_examples))):
        ex = random.choice(rap_examples)
        print(f"\n  --- sample {idx+1} ---")
        print(f"  user:      {ex['messages'][1]['content'][:150]}")
        print(f"  assistant: {ex['messages'][-1]['content'][:200]}")
else:
    print("\nWARNING: No rap examples generated! Check lyrics format above.")

In [ ]:
# --- Load SFT augmentation: chosen-only labeled responses + frontier distillation ---
#
# Two sources stacked into `aug_examples`:
#   1. sft_augment.jsonl — chosen responses from pairwise labels (v11-v14)
#   2. distilled_qwen72b_v16.jsonl — Qwen-72B distilled responses on sampled
#      train.jsonl contexts (v16 addition). Style-filtered (no refusals, ≥1
#      profanity marker) so they reinforce the production persona.
#
# Per-token CE loss has no length-bias problem and no off-policy log-ratio,
# so any positive example is safe to weight up. AUGMENT_REPEAT=2 keeps
# pairwise-chosen at 2× weight; DISTILL_REPEAT=1 keeps distilled at parity
# (high quality but lower diversity per-example → don't overamplify).

import glob as _glob

AUGMENT_REPEAT = 2   # weight multiplier for sft_augment.jsonl
DISTILL_REPEAT = 1   # weight multiplier for distilled_qwen72b_v16.jsonl


def _resolve_aug_path(basename):
    """Find an augmentation file across Kaggle dataset mounts, repo root, eval/ ..."""
    candidates = [
        f"/kaggle/input/businessgpt-eval/{basename}",
        f"/kaggle/input/businessgpt-eval/eval/{basename}",
        f"eval/{basename}",
        f"{basename}",
    ]
    p = next((c for c in candidates if os.path.isfile(c)), None)
    if p is None:
        matches = _glob.glob(f"/kaggle/input/**/{basename}", recursive=True)
        if matches:
            p = matches[0]
            print(f"  Found {basename} via glob: {p}")
    return p


def _load_jsonl_examples(path):
    """Load .jsonl, strip _meta, return list of {messages: [...]} dicts."""
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            ex.pop("_meta", None)
            out.append(ex)
    return out


aug_examples = []

# Source 1: pairwise-chosen augment
_aug_path = _resolve_aug_path("sft_augment.jsonl")
if _aug_path is not None:
    _pairwise = _load_jsonl_examples(_aug_path)
    aug_examples.extend(_pairwise * AUGMENT_REPEAT)
    print(f"Pairwise augment: {len(_pairwise)} unique → {len(_pairwise) * AUGMENT_REPEAT} after x{AUGMENT_REPEAT}")
    print(f"  source: {_aug_path}")
else:
    print("WARNING: sft_augment.jsonl not found — training without pairwise augment")

# Source 2: frontier distillation (v16+)
_distill_path = _resolve_aug_path("distilled_qwen72b_v16.jsonl")
if _distill_path is not None:
    _distilled = _load_jsonl_examples(_distill_path)
    aug_examples.extend(_distilled * DISTILL_REPEAT)
    print(f"Distill augment:  {len(_distilled)} unique → {len(_distilled) * DISTILL_REPEAT} after x{DISTILL_REPEAT}")
    print(f"  source: {_distill_path}")
else:
    print("INFO: distilled_qwen72b_v16.jsonl not found — proceeding as v15-equivalent SFT")

print(f"Total augment examples: {len(aug_examples)}")

## 2. Load Model + LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
import torch

# v15: bumped 2B → 9B. fp16 weights = 18 GB, splits across T4×2 (16 GB each)
# via device_map="auto" → ~9 GB / GPU, ~5 GB headroom for activations + logits + LoRA grads.
#
# MAX_SEQ_LENGTH 4096 → 1024 (was 2048; OOM'd at logits.float() upcast in loss).
# At batch=1: logits = 1 × seq × ~152K × 4B fp32. seq=2048 → 1.2 GB; seq=1024 → 0.6 GB.
# Loss path also peaks during the float() upcast which transiently doubles this. seq=1024
# gives ~2-3 GB headroom on T4 vs the borderline 14.56 GB / GPU 1 limit.
MAX_SEQ_LENGTH = 1024
MODEL_ID = "huihui-ai/Huihui-Qwen3.5-9B-abliterated"

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
except Exception as e:
    print(f"AutoModelForCausalLM failed ({e}), trying AutoModel...")
    model = AutoModel.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

if next(model.parameters()).dtype not in (torch.float16, torch.float32):
    print(f"WARNING: model is {next(model.parameters()).dtype}, casting to float16...")
    model = model.half()

model.gradient_checkpointing_enable()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

CHAT_TEMPLATE_NO_THINK = (
    "{%- for message in messages %}"
    "<|im_start|>{{ message.role }}\n"
    "{{ message.content }}<|im_end|>\n"
    "{%- endfor %}"
    "{%- if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{%- endif %}"
)
tokenizer.chat_template = CHAT_TEMPLATE_NO_THINK

print(f"Model loaded: {model.__class__.__name__}")
print(f"Dtype: {next(model.parameters()).dtype}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

n_gpus = torch.cuda.device_count()
print(f"\nGPUs available: {n_gpus}")
for i in range(n_gpus):
    mem_total = torch.cuda.get_device_properties(i).total_memory / 1024**3
    mem_alloc = torch.cuda.memory_allocated(i) / 1024**3
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {mem_alloc:.1f} / {mem_total:.1f} GB used")

if hasattr(model, "hf_device_map"):
    devices_used = set(str(v) for v in model.hf_device_map.values())
    print(f"  device_map spread across: {devices_used}")
    print(f"  Layers mapped: {len(model.hf_device_map)}")
else:
    print(f"  Model on single device: {next(model.parameters()).device}")

## 3. Apply LoRA + Prepare Dataset

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules="all-linear",
    lora_dropout=0.05,
    use_dora=False,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"LoRA: {trainable:,} / {total_params:,} ({trainable/total_params*100:.1f}%) trainable")
model.print_trainable_parameters()

In [ ]:
def format_chat(messages):
    """Manual chat formatting for inference — avoids <think> block injection."""
    parts = []
    for msg in messages:
        parts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>")
    return "\n".join(parts)

def count_tokens(example):
    text = format_chat(example["messages"])
    return len(tokenizer.encode(text))

# Filter chat examples by token length
chat_filtered = [ex for ex in all_examples if count_tokens(ex) <= MAX_SEQ_LENGTH]
chat_dropped = len(all_examples) - len(chat_filtered)
print(f"Chat after token filter: {len(chat_filtered)} / {len(all_examples)} ({chat_dropped} dropped)")

# Filter rap examples by token length
rap_filtered = [ex for ex in rap_examples if count_tokens(ex) <= MAX_SEQ_LENGTH]
rap_dropped = len(rap_examples) - len(rap_filtered)
print(f"Rap after token filter:  {len(rap_filtered)} / {len(rap_examples)} ({rap_dropped} dropped)")

# Deduplicate by full message hash (preserves augmentation variants)
import hashlib

def _dedup(examples):
    seen = set()
    out = []
    for ex in examples:
        key = "|||".join(m["role"] + ":" + m["content"] for m in ex["messages"])
        h = hashlib.md5(key.encode()).hexdigest()
        if h not in seen:
            seen.add(h)
            out.append(ex)
    return out

chat_deduped = _dedup(chat_filtered)
rap_deduped = _dedup(rap_filtered)
print(f"After dedup: chat {len(chat_filtered)} -> {len(chat_deduped)}, "
      f"rap {len(rap_filtered)} -> {len(rap_deduped)}")

# Augment: filter by token length, but DO NOT dedup — replications carry weight.
aug_filtered = [ex for ex in aug_examples if count_tokens(ex) <= MAX_SEQ_LENGTH]
aug_dropped = len(aug_examples) - len(aug_filtered)
print(f"Aug after token filter:  {len(aug_filtered)} / {len(aug_examples)} ({aug_dropped} dropped)")

final = chat_deduped + rap_deduped + aug_filtered
print(f"\nCombined: {len(final)} (chat: {len(chat_deduped)}, rap: {len(rap_deduped)}, "
      f"aug: {len(aug_filtered)}, "
      f"rap ratio: {len(rap_deduped)/len(final)*100:.1f}%, "
      f"aug ratio: {len(aug_filtered)/len(final)*100:.1f}%)")

import random
random.seed(42)
shuffled = final[:]
random.shuffle(shuffled)

val_size = max(50, int(len(shuffled) * 0.05))
val_data = shuffled[:val_size]
train_data = shuffled[val_size:]

print(f"Train: {len(train_data)}, Val: {len(val_data)}")

In [ ]:
from datasets import Dataset

im_start_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
assistant_nl_ids = tokenizer.encode("assistant\n", add_special_tokens=False)
RESPONSE_PREFIX = [im_start_id] + assistant_nl_ids
print(f"Response prefix tokens ({len(RESPONSE_PREFIX)}): {RESPONSE_PREFIX}")
print(f"  Decoded: {tokenizer.decode(RESPONSE_PREFIX)!r}")

def tokenize_with_labels(example):
    """Tokenize full conversation; loss only on LAST assistant turn."""
    msgs = example["messages"]
    parts = []
    for msg in msgs:
        parts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>")
    full_text = "\n".join(parts)

    encoded = tokenizer(full_text, truncation=True, max_length=MAX_SEQ_LENGTH,
                        add_special_tokens=False)
    input_ids = encoded["input_ids"]

    labels = [-100] * len(input_ids)
    prefix_len = len(RESPONSE_PREFIX)

    last_match = -1
    for i in range(len(input_ids) - prefix_len + 1):
        if input_ids[i:i + prefix_len] == RESPONSE_PREFIX:
            last_match = i + prefix_len

    if last_match >= 0:
        labels[last_match:] = input_ids[last_match:]

    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}

train_ds = Dataset.from_list(train_data).map(tokenize_with_labels, remove_columns=["messages"])
val_ds = Dataset.from_list(val_data).map(tokenize_with_labels, remove_columns=["messages"])

n_empty = sum(1 for ex in train_ds if all(l == -100 for l in ex["labels"]))
active_counts = [sum(1 for l in ex["labels"] if l != -100) for ex in train_ds]
print(f"\nTrain: {len(train_ds)}, Val: {len(val_ds)}")
print(f"Active labels per example: min={min(active_counts)}, "
      f"median={sorted(active_counts)[len(active_counts)//2]}, "
      f"max={max(active_counts)}, mean={sum(active_counts)/len(active_counts):.1f}")
print(f"Examples with zero active labels: {n_empty} / {len(train_ds)}")
assert n_empty == 0, f"BUG: {n_empty} examples have no active labels — response prefix not found!"

In [ ]:
# Save val examples for use in bench notebook (messages format)
import json as _json

val_export_path = "val_examples.json"
with open(val_export_path, "w", encoding="utf-8") as f:
    _json.dump(val_data, f, ensure_ascii=False, indent=1)

print(f"Saved {len(val_data)} val examples to {val_export_path}")
print(f"  (download from Kaggle output or /kaggle/working/{val_export_path})")
print(f"  Format: messages (keys: {list(val_data[0].keys())})")

## 4. Train

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, TrainerCallback
import torch

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# v15 (9B) memory + time tuning:
# - NUM_EPOCHS 2 → 1: previous 2-epoch run timed out at 12h Kaggle limit.
#   9k+1.2k aug examples with NEFTune is enough for one pass at this scale.
# - GRAD_ACCUM 16: keeps effective batch=16, halves per-step memory pressure.
# - optim="adamw_8bit": LoRA optimizer states quantized to 8-bit via bitsandbytes.
NUM_EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 16
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
STEPS_PER_EPOCH = len(train_ds) // EFFECTIVE_BATCH
EVAL_STEPS = max(50, STEPS_PER_EPOCH // 4)  # ~4 evals per epoch, min 50 to avoid eval spam
TOTAL_STEPS = STEPS_PER_EPOCH * NUM_EPOCHS

print(f"Training config:")
print(f"  Examples: {len(train_ds)}")
print(f"  Batch size: {BATCH_SIZE} x {GRAD_ACCUM} grad_accum = {EFFECTIVE_BATCH} effective")
print(f"  Steps/epoch: {STEPS_PER_EPOCH}")
print(f"  Total steps: {TOTAL_STEPS} ({NUM_EPOCHS} epochs)")
print(f"  Eval every: {EVAL_STEPS} steps")
print(f"  Completion-only loss: ON (manual label masking, bypasses TRL)")

model.train()

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
)


# --- HF checkpoint streaming ---
# Pushes the current LoRA adapter to HF every N steps. If Kaggle times out,
# we can still load this partial checkpoint instead of losing the whole run.
# Reuses HF_REPO from the push cell (cell 36ba67fd) — make sure that cell's
# HF_REPO is set BEFORE training starts. Or hardcode here.
_STREAM_HF_REPO = "vXofi/businessgpt-v16-qwen3.5-9b"  # mirror of HF_REPO from push cell
STREAM_EVERY = max(EVAL_STEPS, 100)  # push at least every 100 steps

class _HFStreamCheckpoint(TrainerCallback):
    """Push LoRA adapter to HF every STREAM_EVERY steps. Survives Kaggle timeout."""

    def __init__(self, repo_id, every_n_steps, tokenizer):
        from huggingface_hub import HfApi
        self.repo_id = repo_id
        self.every = every_n_steps
        self.tokenizer = tokenizer
        self.api = HfApi()
        self.api.create_repo(repo_id, exist_ok=True)
        self._tmp_dir = "/kaggle/working/_streaming_ckpt"

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step == 0 or state.global_step % self.every != 0:
            return
        if model is None:
            return
        try:
            os.makedirs(self._tmp_dir, exist_ok=True)
            model.save_pretrained(self._tmp_dir, safe_serialization=True)
            self.tokenizer.save_pretrained(self._tmp_dir)
            self.api.upload_folder(
                folder_path=self._tmp_dir,
                repo_id=self.repo_id,
                commit_message=f"Streaming checkpoint @ step {state.global_step}/{state.max_steps}",
            )
            print(f"  [HF stream] pushed step {state.global_step}/{state.max_steps}")
        except Exception as e:
            print(f"  [HF stream] push failed (non-fatal): {e}")


trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    callbacks=[_HFStreamCheckpoint(_STREAM_HF_REPO, STREAM_EVERY, tokenizer)],
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=50,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=5e-5,
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=EVAL_STEPS,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        fp16=True,
        bf16=False,
        gradient_checkpointing=True,
        neftune_noise_alpha=5,
    ),
)

# --- Dry run: real forward pass to verify loss is non-zero BEFORE 6+ hours of training ---
_dry_batch = collator([train_ds[0]])
_input_dev = next(model.parameters()).device
_dry_batch = {k: v.to(_input_dev) for k, v in _dry_batch.items()}
with torch.no_grad():
    _dry_out = model(**_dry_batch)
_dry_loss = _dry_out.loss.item()
print(f"\nDry-run forward pass loss: {_dry_loss:.4f}")
assert _dry_loss > 0 and not torch.isnan(_dry_out.loss), \
    f"STOP: dry-run loss is {_dry_loss} — training would waste hours!"
print("Dry run passed — training will produce real gradients.")

# Free dry-run memory before training kicks in.
del _dry_batch, _dry_out
torch.cuda.empty_cache()

In [ ]:
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Final loss: {trainer_stats.training_loss:.4f}")

## 5. Save Model

In [ ]:
SAVE_DIR = "businessgpt_v16_9b_model"
model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Full model saved to {SAVE_DIR}/")

for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f"{SAVE_DIR}/{f}")
    if size > 1024 * 1024:
        print(f"  {f}: {size/1024/1024:.1f} MB")
    else:
        print(f"  {f}: {size/1024:.1f} KB")

## 6. Test Inference

In [ ]:
import re

model.eval()

_input_device = next(model.parameters()).device

_im_start_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
_im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
_eos_ids = [_im_end_id]
if _im_start_id is not None and _im_start_id != tokenizer.unk_token_id:
    _eos_ids.append(_im_start_id)

def chat(messages_history, max_new_tokens=200):
    """Inference supporting both flat context and multi-turn messages."""
    if messages_history and isinstance(messages_history[0], str):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "\n".join(messages_history)},
        ]
    else:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        messages.extend(messages_history)

    input_text = format_chat(messages) + "\n<|im_start|>assistant\n"

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(_input_device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.95,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.1,
            eos_token_id=_eos_ids,
        )

    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    generated = re.sub(r"<think>.*?</think>\s*", "", generated, flags=re.DOTALL).strip()
    generated = re.sub(r"<\|im_start\|>.*", "", generated, flags=re.DOTALL)
    generated = re.sub(r"<\|im_end\|>", "", generated)
    generated = re.sub(r"^<bot>\s*", "", generated).strip()
    print(generated)
    return generated

In [ ]:
# Test 1: Use real context from validation set
print("=" * 60)
print("Test 1: Real context from validation data")
print("=" * 60)
test_ex = val_data[0]
ctx_lines = test_ex["messages"][1]["content"].split("\n")
print("Context:")
for line in ctx_lines[-5:]:
    print(f"  {line[:100]}")
print(f"\nExpected: {test_ex['messages'][-1]['content'][:100]}")
print(f"Generated:")
response = chat(ctx_lines)

In [ ]:
# Test 2: Custom prompt
print("=" * 60)
print("Test 2: Custom context")
print("=" * 60)
custom_context = [
    "кто сделал 15 практику?",
    "я не делал",
    "я тоже нет",
    "может кто-нибудь уже начнёт",
    "ну давайте",
    "мне лень",
    "аналогично",
    "я могу попробовать",
    "давай мельник",
    "наш герой",
]
print("Context:")
for line in custom_context:
    print(f"  {line}")
print(f"\nGenerated:")
response = chat(custom_context)

In [ ]:
chat(["сосал?"])

In [ ]:
# Test 3: Multiple generations to check diversity
print("=" * 60)
print("Test 3: Multiple generations (same context)")
print("=" * 60)
for i in range(5):
    print(f"\nGeneration {i+1}:")
    response = chat(custom_context, max_new_tokens=64)

## 7. Export for Deployment (Optional)

Merge LoRA weights into the base model and save as a full model, or push to HuggingFace Hub.

In [ ]:
# Model already saved in cell above as SAVE_DIR
# GGUF conversion can be done locally with llama.cpp after downloading

In [ ]:
# GGUF Q8_0 conversion: run locally after downloading the model
# python3 llama.cpp/convert_hf_to_gguf.py businessgpt_v6_model/ --outfile v6-f16.gguf --outtype f16
# ./llama.cpp/build/bin/llama-quantize v6-f16.gguf v6-Q8_0.gguf Q8_0


In [ ]:
from huggingface_hub import HfApi

HF_REPO = "vXofi/businessgpt-v16-qwen3.5-9b"
api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id=HF_REPO,
    commit_message="Upload BusinessGPT v16 LoRA (Qwen3.5-9B-abliterated base + chat + rap + sft_augment + Qwen-72B distilled augment)",
)
print(f"Pushed to https://huggingface.co/{HF_REPO}")

## 8. Eval — DISABLED for 9B

9B inference on T4×2 takes ~3-4× longer than 2B. Running 4 candidates × 631 prompts
inline would push total notebook time past Kaggle's 12 h budget. Eval is moved to
the standalone `eval_only.ipynb` notebook which:

- Loads v15 LoRA from HF (this notebook just pushed it)
- Has fresh 12 h budget for inference only
- Checkpoints to HF every 50 prompts so timeouts are recoverable

After this notebook finishes, open a fresh Kaggle notebook with `eval_only.ipynb`,
set `LORA_REPO = "vXofi/businessgpt-v15-qwen3.5-9b"`, and Save & Run All.

The cell below is gated by `RUN_EVAL_AT_END = False`. Flip to True only if you
have unused budget on the training notebook AND the model is small.

In [ ]:
RUN_EVAL_AT_END = False  # 9B inference too slow for inline eval; use eval_only.ipynb

if not RUN_EVAL_AT_END:
    print("Skipping eval — RUN_EVAL_AT_END=False.")
    print(f"To run eval: open `eval_only.ipynb`, set LORA_REPO={'vXofi/businessgpt-v15-qwen3.5-9b'!r}, "
          f"attach `businessgpt-eval` Kaggle dataset, Save & Run All.")
else:
    import json
    import os
    import re as _re
    import io
    import contextlib
    import glob as _glob
    import torch

    EVAL_DIR = "/kaggle/working/eval"
    os.makedirs(EVAL_DIR, exist_ok=True)

    _m = _re.search(r"(v\d+)", SAVE_DIR)
    VERSION = _m.group(1) if _m else "vX"

    _candidates = [
        "/kaggle/input/businessgpt-eval/golden_prompts.json",
        "/kaggle/input/businessgpt-eval/eval/golden_prompts.json",
        "eval/golden_prompts.json",
        "golden_prompts.json",
    ]
    _golden_path = next((p for p in _candidates if os.path.isfile(p)), None)
    if _golden_path is None:
        _matches = _glob.glob("/kaggle/input/**/golden_prompts.json", recursive=True)
        if _matches:
            _golden_path = _matches[0]
    if _golden_path is None:
        from huggingface_hub import hf_hub_download
        _golden_path = hf_hub_download(repo_id=HF_REPO, filename="eval/golden_prompts.json")

    with open(_golden_path, encoding="utf-8") as f:
        golden = json.load(f)
    print(f"Loaded {len(golden)} prompts")

    CANDIDATE_PARAMS = [
        {"idx": 0, "temperature": 0.7,  "top_p": 0.9,  "top_k": 50, "repetition_penalty": 1.1, "seed": 42},
        {"idx": 1, "temperature": 0.95, "top_p": 0.9,  "top_k": 50, "repetition_penalty": 1.1, "seed": 43},
        {"idx": 2, "temperature": 1.1,  "top_p": 0.92, "top_k": 60, "repetition_penalty": 1.1, "seed": 44},
        {"idx": 3, "temperature": 1.3,  "top_p": 0.95, "top_k": 80, "repetition_penalty": 1.05, "seed": 45},
    ]
    DEFAULT_CANDIDATE_IDX = 1
    MAX_NEW_TOKENS = 200

    model.eval()

    def _chat_with_params(context, max_new_tokens, temperature, top_p, top_k,
                          repetition_penalty, seed):
        if context and isinstance(context[0], str):
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": "\n".join(context)},
            ]
        else:
            messages = [{"role": "system", "content": SYSTEM_PROMPT}]
            messages.extend(context)

        input_text = format_chat(messages) + "\n<|im_start|>assistant\n"
        inputs = tokenizer(input_text, return_tensors="pt", add_special_tokens=False).to(_input_device)

        torch.manual_seed(seed)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                repetition_penalty=repetition_penalty,
                eos_token_id=_eos_ids,
            )
        generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
        generated = _re.sub(r"<think>.*?</think>\s*", "", generated, flags=_re.DOTALL).strip()
        generated = _re.sub(r"<\|im_start\|>.*", "", generated, flags=_re.DOTALL)
        generated = _re.sub(r"<\|im_end\|>", "", generated)
        generated = _re.sub(r"^<bot>\s*", "", generated).strip()
        return generated

    multi = []
    for i, p in enumerate(golden):
        candidates = []
        for params in CANDIDATE_PARAMS:
            response = _chat_with_params(
                p["context"], max_new_tokens=MAX_NEW_TOKENS,
                temperature=params["temperature"], top_p=params["top_p"],
                top_k=params["top_k"], repetition_penalty=params["repetition_penalty"],
                seed=params["seed"],
            )
            candidates.append({**params, "response": response})

        multi.append({
            "id": p["id"], "category": p["category"],
            "held_out": p.get("held_out", False), "context": p["context"],
            "version": VERSION, "candidates": candidates,
        })
        if (i + 1) % 25 == 0 or (i + 1) == len(golden):
            print(f"  {i+1}/{len(golden)}")

    multi_path = os.path.join(EVAL_DIR, f"generations_{VERSION}_multi.json")
    with open(multi_path, "w", encoding="utf-8") as f:
        json.dump(multi, f, ensure_ascii=False, indent=1)
    print(f"\nSaved {len(multi)} multi-candidate generations to {multi_path}")

    single = []
    for entry in multi:
        default = next(c for c in entry["candidates"] if c["idx"] == DEFAULT_CANDIDATE_IDX)
        single.append({
            "id": entry["id"], "category": entry["category"],
            "held_out": entry["held_out"], "context": entry["context"],
            "response": default["response"], "version": VERSION,
            "sampling_params": {k: default[k] for k in ("temperature", "top_p", "top_k", "repetition_penalty")},
            "seed": default["seed"], "sampling_params_extra": {"max_new_tokens": MAX_NEW_TOKENS},
        })
    single_path = os.path.join(EVAL_DIR, f"generations_{VERSION}.json")
    with open(single_path, "w", encoding="utf-8") as f:
        json.dump(single, f, ensure_ascii=False, indent=1)

    for path, label in [(single_path, "single"), (multi_path, "multi")]:
        try:
            api.upload_file(
                path_or_fileobj=path,
                path_in_repo=f"eval/{os.path.basename(path)}",
                repo_id=HF_REPO,
                commit_message=f"Upload eval generations ({VERSION}, {label})",
            )
            print(f"  HF upload OK: {os.path.basename(path)}")
        except Exception as e:
            print(f"  HF upload failed for {label}: {e}")